# Aegis Alpha - Phase 2: Exploratory Data Analysis (EDA)

This notebook validates the ingested SQLite dataset (`data/market_data.db`), builds the 40 numeric and 57 dummy features defined in `src/features.py`, and conducts target distribution, missing-value, regime, and feature correlation analysis against `TO_PREDICT` (`is_positive_growth_4h`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DB_PATH, DATA_DIR, SYMBOLS
from src.features import (
    TO_PREDICT,
    TARGET_COL,
    NUMERIC,
    DUMMIES,
    FEATURE_COLS,
    load_raw_tables_from_sqlite,
    build_feature_dataframe,
    load_dataframe,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
print(f'Project root     : {PROJECT_ROOT}')
print(f'Target column    : {TO_PREDICT}')
print(f'NUMERIC features : {len(NUMERIC)}')
print(f'DUMMIES features : {len(DUMMIES)}')
print(f'Total features   : {len(FEATURE_COLS)}')

## 1. Data Ingestion Audit and Feature Generation

Loads raw tables from `data/market_data.db` (or falls back to `data/offline_snapshot.parquet` if running in offline peer-review mode).

In [ ]:
offline_parquet = DATA_DIR / 'offline_snapshot.parquet'
if DB_PATH.exists():
    df_ohlcv, df_fg = load_raw_tables_from_sqlite(DB_PATH)
    print(f'Raw OHLCV rows        : {len(df_ohlcv):,}')
    print(f'Raw Fear & Greed rows : {len(df_fg):,}')
    display(df_ohlcv.groupby('symbol')['datetime_utc'].agg(['count', 'min', 'max']))
    df_raw_features = build_feature_dataframe(
        df_ohlcv, df_fg, drop_na_features=False, drop_na_target=False
    )
    df = df_raw_features.dropna(subset=NUMERIC + TO_PREDICT).copy()
    df[TARGET_COL] = df[TARGET_COL].astype(int)
elif offline_parquet.exists():
    df_raw_features = load_dataframe(offline_parquet)
    df = df_raw_features.dropna(subset=NUMERIC + TO_PREDICT).copy()
    df[TARGET_COL] = df[TARGET_COL].astype(int)
    print(f'Loaded offline snapshot: {len(df):,} rows')
else:
    raise FileNotFoundError(
        'Neither data/market_data.db nor data/offline_snapshot.parquet found. '
        'Run python -m src.data_fetcher first.'
    )

print(f'Clean modeling rows: {len(df):,}')

## 2. Missing-Value and Data Quality Checks

In [ ]:
na_before = df_raw_features[NUMERIC + TO_PREDICT].isna().sum()
na_before_pct = (na_before / len(df_raw_features) * 100.0).round(4)
na_report = pd.DataFrame({'missing_raw': na_before, 'missing_raw_pct': na_before_pct})
print('Top 15 columns with warmup/tail NaNs prior to dropna:')
display(na_report.sort_values('missing_raw', ascending=False).head(15))

na_after = df[FEATURE_COLS + TO_PREDICT].isna().sum().sum()
dup_keys = df.duplicated(subset=['symbol', 'timestamp']).sum()
print(f'Remaining NaNs after cleaning : {na_after}')
print(f'Duplicate (symbol, timestamp) : {dup_keys}')

## 3. Target Variable (`TO_PREDICT`) Distribution Analysis

Inspects class balance for `is_positive_growth_4h` (1 when forward 4h return >= +0.5%) overall, per asset, per calendar year, and per sentiment regime.

In [ ]:
df['year'] = df['datetime'].dt.year

print('Overall target distribution:')
display(df[TARGET_COL].value_counts(normalize=True).rename('proportion').to_frame())

print('Target positive rate and mean 4h return by symbol:')
by_symbol = df.groupby('symbol').agg(
    rows=(TARGET_COL, 'count'),
    positive_rate=(TARGET_COL, 'mean'),
    mean_ret_4h=('future_return_4h', 'mean'),
    std_ret_4h=('future_return_4h', 'std'),
)
display(by_symbol)

print('Target positive rate by year:')
display(df.groupby('year')[TARGET_COL].agg(['count', 'mean']))

print('Target positive rate by Fear & Greed sentiment regime:')
by_sent = df.groupby('sentiment_regime').agg(
    rows=(TARGET_COL, 'count'),
    positive_rate=(TARGET_COL, 'mean'),
    mean_ret_4h=('future_return_4h', 'mean'),
).sort_values('positive_rate', ascending=False)
display(by_sent)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
by_symbol['positive_rate'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Target Positive Rate by Symbol')
axes[0].set_ylabel('P(future_return_4h >= 0.5%)')
axes[0].axhline(df[TARGET_COL].mean(), color='crimson', linestyle='--')

by_sent['positive_rate'].plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Target Positive Rate by Sentiment Regime')
axes[1].set_ylabel('P(future_return_4h >= 0.5%)')
axes[1].axhline(df[TARGET_COL].mean(), color='crimson', linestyle='--')
plt.tight_layout()
plt.show()

## 4. Feature Correlation Analysis Against `TO_PREDICT`

Evaluates linear (Pearson) and rank (Spearman) correlations between all numeric features and both `is_positive_growth_4h` and continuous `future_return_4h`.

In [ ]:
pearson_target = df[NUMERIC].corrwith(df[TARGET_COL]).rename('pearson_vs_target')
pearson_ret = df[NUMERIC].corrwith(df['future_return_4h']).rename('pearson_vs_ret_4h')
corr_summary = pd.concat([pearson_target, pearson_ret], axis=1)
corr_summary['abs_corr_target'] = corr_summary['pearson_vs_target'].abs()
corr_summary = corr_summary.sort_values('abs_corr_target', ascending=False)

print('Top 20 numeric features correlated with TO_PREDICT:')
display(corr_summary.head(20))

top_feats = corr_summary.head(15).index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
corr_summary.head(15)['pearson_vs_target'].sort_values().plot(
    kind='barh', ax=axes[0], color='slategray'
)
axes[0].set_title('Top 15 Feature Correlations with is_positive_growth_4h')
axes[0].axvline(0.0, color='black', linewidth=0.8)

sns.heatmap(
    df[top_feats].corr(),
    cmap='coolwarm',
    center=0.0,
    ax=axes[1],
    cbar_kws={'shrink': 0.8},
)
axes[1].set_title('Inter-Feature Correlation Matrix (Top 15 Numeric Features)')
plt.tight_layout()
plt.show()

## 5. Volatility Regime & Contrarian Fear Filter Validation

In [ ]:
vol_table = df.groupby('volatility_regime').agg(
    rows=(TARGET_COL, 'count'),
    positive_rate=(TARGET_COL, 'mean'),
    mean_ret_4h=('future_return_4h', 'mean'),
    tail_loss_p05=('future_return_4h', lambda s: s.quantile(0.05)),
)
display(vol_table)

cross_pivot = pd.pivot_table(
    df,
    values='future_return_4h',
    index='sentiment_regime',
    columns='volatility_regime',
    aggfunc='mean',
)
print('Mean 4h Forward Return by Sentiment Regime x Volatility Regime:')
display(cross_pivot)